# AI in Molecular Biology — Promoter Prediction

This notebook builds a DNA dataset, then trains and compares three models (Logistic Regression, Random Forest, 1D-CNN) to detect bacterial-type promoters.

**Run all cells in order.**

Install once (uncomment if needed):

In [ ]:
# !pip install numpy pandas scikit-learn matplotlib tensorflow

In [1]:
!pip install numpy pandas scikit-learn matplotlib tensorflow

## 1. Create the dataset

In [2]:
"""
Dataset generator: promoters (label 1) vs three kinds of non-promoters (label 0):
random DNA, single-box decoys, and wrong-spacing decoys.
"""
import random, csv
random.seed(42)
BASES = "ACGT"
L = 100
N_PER_CLASS = 4000
MUT = 0.10

def rnd(n): return "".join(random.choice(BASES) for _ in range(n))

def mutate(m, rate=MUT):
    return "".join(random.choice([b for b in BASES if b != c]) if random.random() < rate else c for c in m)

def embed(core):
    left = random.randint(3, L - len(core) - 3)
    return rnd(left) + core + rnd(L - len(core) - left)

def promoter():
    return embed(mutate("TTGACA") + rnd(random.randint(16, 18)) + mutate("TATAAT"))

def decoy_single():
    box = mutate(random.choice(["TTGACA", "TATAAT"]))
    return embed(box)

def decoy_spacing():
    gap = random.choice(list(range(4, 11)) + list(range(26, 36)))
    return embed(mutate("TTGACA") + rnd(gap) + mutate("TATAAT"))

rows = []
for _ in range(N_PER_CLASS):
    rows.append((promoter(), 1))
    r = random.random()
    if r < 0.5:    rows.append((rnd(L), 0))
    elif r < 0.75: rows.append((decoy_single(), 0))
    else:          rows.append((decoy_spacing(), 0))
random.shuffle(rows)

with open("promoter_dataset.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["sequence", "label"]); w.writerows(rows)
print("Saved promoter_dataset.csv :", len(rows), "sequences")


Saved promoter_dataset.csv : 8000 sequences


## 2. Load data and inspect

In [3]:
import pandas as pd
df = pd.read_csv("promoter_dataset.csv")
print(df.shape)
df.head()

(8000, 2)


,sequence,label
0,CTCGAGTCCAGGAAGGTATTAAAATGTGCGACCCGTTGACAGGCGC...,0
1,TGGTATATGTCCTGAGTACTTGAGTTCCGAGAGTAGCACGATCCTA...,0
2,TTCGGTCGGGTCGAAGGCTCCCTTGCGTGAGGAAGATTCCGCGGAA...,0
3,AATTAAAAGGAAAGTATCGGCATGGGTAACGCAGTGAAGTGATGGG...,0
4,TTAATAGGGGAGCTCCTATTCTCTGATGTGTGCAGTCGGTTGGTGC...,0


In [4]:
df["label"].value_counts()

,count
label,
0,4000
1,4000


## 3. Imports and setup

In [5]:
import os, random
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             matthews_corrcoef, roc_auc_score, roc_curve,
                             confusion_matrix, ConfusionMatrixDisplay)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

## 4. Encode sequences (one-hot)

In [6]:
df = pd.read_csv("promoter_dataset.csv")
seqs, y = df["sequence"].values, df["label"].values
idx = {"A": 0, "C": 1, "G": 2, "T": 3}

def one_hot(s):
    m = np.zeros((len(s), 4), dtype=np.float32)
    for i, ch in enumerate(s):
        m[i, idx[ch]] = 1.0
    return m

X_oh = np.stack([one_hot(s) for s in seqs])            # (N, 100, 4)
print("One-hot tensor shape:", X_oh.shape)

One-hot tensor shape: (8000, 100, 4)


## 5. Train / validation / test split

In [7]:
i_tr, i_tmp = train_test_split(np.arange(len(y)), test_size=0.30, stratify=y, random_state=SEED)
i_va, i_te = train_test_split(i_tmp, test_size=0.50, stratify=y[i_tmp], random_state=SEED)
print(f"Train {len(i_tr)} | Val {len(i_va)} | Test {len(i_te)}")

Train 5600 | Val 1200 | Test 1200


## 6. Baseline models (k-mer + Logistic Regression / Random Forest)

In [8]:
K = 6
cv = CountVectorizer(analyzer="char", ngram_range=(K, K), lowercase=False)
Xk_tr = cv.fit_transform(seqs[i_tr]).toarray()
Xk_te = cv.transform(seqs[i_te]).toarray()

scores, probas = {}, {}
def evaluate(name, y_true, prob):
    pred = (prob >= 0.5).astype(int)
    scores[name] = dict(Accuracy=accuracy_score(y_true, pred), Precision=precision_score(y_true, pred),
                        Recall=recall_score(y_true, pred), F1=f1_score(y_true, pred),
                        MCC=matthews_corrcoef(y_true, pred), AUC=roc_auc_score(y_true, prob))
    probas[name] = prob

lr = LogisticRegression(max_iter=3000).fit(Xk_tr, y[i_tr])
evaluate("LogReg (6-mer)", y[i_te], lr.predict_proba(Xk_te)[:, 1])
rf = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1).fit(Xk_tr, y[i_tr])
evaluate("RandomForest (6-mer)", y[i_te], rf.predict_proba(Xk_te)[:, 1])

## 7. Build and train the 1D-CNN

In [9]:
def build_cnn():
    m = models.Sequential([
        layers.Input(shape=(100, 4)),
        layers.Conv1D(32, 8, activation="relu", padding="same", name="conv1"),
        layers.MaxPooling1D(2),
        layers.Conv1D(64, 8, activation="relu", padding="same"),
        layers.MaxPooling1D(2),
        layers.Conv1D(64, 8, activation="relu", padding="same"),
        layers.GlobalMaxPooling1D(),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])
    return m

cnn = build_cnn(); cnn.summary()
hist = cnn.fit(X_oh[i_tr], y[i_tr], validation_data=(X_oh[i_va], y[i_va]),
               epochs=60, batch_size=64, verbose=2,
               callbacks=[callbacks.EarlyStopping(patience=8, restore_best_weights=True)])
evaluate("CNN (one-hot)", y[i_te], cnn.predict(X_oh[i_te], verbose=0).ravel())

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1 (Conv1D)                  │ (None, 100, 32)        │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 50, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 50, 64)         │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 25, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 25, 64)         │        32,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 64)             │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 52,449 (204.88 KB)

 Trainable params: 52,449 (204.88 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/60
88/88 - 12s - 134ms/step - accuracy: 0.5470 - loss: 0.6877 - val_accuracy: 0.6342 - val_loss: 0.6543
Epoch 2/60
88/88 - 0s - 4ms/step - accuracy: 0.7093 - loss: 0.5508 - val_accuracy: 0.8483 - val_loss: 0.3372
Epoch 3/60
88/88 - 0s - 4ms/step - accuracy: 0.8668 - loss: 0.3192 - val_accuracy: 0.8767 - val_loss: 0.2885
Epoch 4/60
88/88 - 0s - 4ms/step - accuracy: 0.9093 - loss: 0.2398 - val_accuracy: 0.9050 - val_loss: 0.2445
Epoch 5/60
88/88 - 0s - 4ms/step - accuracy: 0.9261 - loss: 0.2025 - val_accuracy: 0.9058 - val_loss: 0.2463
Epoch 6/60
88/88 - 0s - 5ms/step - accuracy: 0.9443 - loss: 0.1670 - val_accuracy: 0.8958 - val_loss: 0.2578
Epoch 7/60
88/88 - 0s - 5ms/step - accuracy: 0.9554 - loss: 0.1386 - val_accuracy: 0.8992 - val_loss: 0.2685
Epoch 8/60
88/88 - 1s - 6ms/step - accuracy: 0.9654 - loss: 0.1168 - val_accuracy: 0.8933 - val_loss: 0.3021
Epoch 9/60
88/88 - 0s - 5ms/step - accuracy: 0.9712 - loss: 0.0997 - val_accuracy: 0.9042 - val_loss: 0.2708
Epoch 10/60
88/8

## 8. Results table

In [10]:
table = pd.DataFrame(scores).T.round(3)
print("\nTEST SET RESULTS\n", table, "\n")
table.to_csv("results.csv")


TEST SET RESULTS
                       Accuracy  Precision  Recall     F1    MCC    AUC
LogReg (6-mer)           0.602      0.601   0.607  0.604  0.203  0.655
RandomForest (6-mer)     0.707      0.680   0.782  0.727  0.418  0.774
CNN (one-hot)            0.906      0.911   0.900  0.905  0.812  0.969 



## 9. Figures

In [11]:
# 6a training curves
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
ax[0].plot(hist.history["loss"], label="train"); ax[0].plot(hist.history["val_loss"], label="validation")
ax[0].set_title("Loss (binary cross-entropy)"); ax[0].set_xlabel("Epoch"); ax[0].legend()
ax[1].plot(hist.history["accuracy"], label="train"); ax[1].plot(hist.history["val_accuracy"], label="validation")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("Epoch"); ax[1].legend()
plt.tight_layout(); plt.savefig("training_curves.png", dpi=150); plt.close()

In [12]:
# 6b ROC curves
plt.figure(figsize=(5, 4.5))
for name, p in probas.items():
    fpr, tpr, _ = roc_curve(y[i_te], p)
    plt.plot(fpr, tpr, label=f"{name} (AUC={scores[name]['AUC']:.3f})")
plt.plot([0, 1], [0, 1], "k--", lw=0.8)
plt.xlabel("False positive rate"); plt.ylabel("True positive rate"); plt.title("ROC curves (test set)")
plt.legend(fontsize=8); plt.tight_layout(); plt.savefig("roc_curves.png", dpi=150); plt.close()

In [13]:
# 6c confusion matrix (CNN)
pred = (probas["CNN (one-hot)"] >= 0.5).astype(int)
ConfusionMatrixDisplay(confusion_matrix(y[i_te], pred), display_labels=["Non-promoter", "Promoter"]).plot(cmap="Blues")
plt.title("Confusion matrix - CNN"); plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight"); plt.close()

In [14]:
# 6d first-layer filters: which filters respond most to a true promoter?
W = cnn.get_layer("conv1").get_weights()[0]            # (8, 4, 32)
strength = np.abs(W).sum(axis=(0, 1))
top = np.argsort(strength)[::-1][:6]
fig, axs = plt.subplots(2, 3, figsize=(9, 4.5))
for a, f in zip(axs.ravel(), top):
    a.imshow(W[:, :, f].T, cmap="RdBu_r", aspect="auto", vmin=-np.abs(W).max(), vmax=np.abs(W).max())
    a.set_yticks(range(4)); a.set_yticklabels(list("ACGT")); a.set_title(f"Filter {f}", fontsize=9)
    a.set_xlabel("Position in filter")
plt.suptitle("Learned first-layer convolution filters (red = positive weight)")
plt.tight_layout(); plt.savefig("cnn_filters.png", dpi=150); plt.close()

cnn.save("promoter_cnn.keras")
print("Saved: results.csv, training_curves.png, roc_curves.png, confusion_matrix.png, cnn_filters.png, promoter_cnn.keras")

Saved: results.csv, training_curves.png, roc_curves.png, confusion_matrix.png, cnn_filters.png, promoter_cnn.keras


## 10. Try the model on new sequences

In [15]:
def predict(seq):
    return float(cnn.predict(one_hot(seq)[None], verbose=0)[0, 0])
promoter_like = "ACGTAGCTAGCTAGGATCGATCGTTGACAGCTAGCTAGCTAGGCTATAATCGATCGATCGTAGCTAGCTAGCTAGCTGATCGATCGATCGATCGTAGCTAG"
wrong_spacing = "ACGTAGCTAGCTAGGATCGATCGTTGACAGCTAGCTATAATCGATCGGGCTAGCGATCGATCGTAGCTAGCTAGCTAGCTGATCGATCGATCGATCGTAGCTAG"
for name, s in [("promoter-like", promoter_like), ("wrong spacing", wrong_spacing)]:
    print(f"{name:14s} -> P(promoter) = {predict(s[:100].ljust(100,'A')):.3f}")

promoter-like  -> P(promoter) = 0.997
wrong spacing  -> P(promoter) = 0.016
